# Jacobian Lens (Hard)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wusche1/Illiad_Mech_Interp/blob/main/exercises/05_jlens/notebook_hard.ipynb)

In [ ]:
import os, importlib

if os.getenv('COLAB_RELEASE_TAG'):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/wusche1/Illiad_Mech_Interp/main/exercises/05_jlens/utils.py",
        "utils.py"
    )
    %pip install "transformer_lens==3.9.0" einops matplotlib -q

import utils
importlib.reload(utils)
from utils import test_run_from_layer, test_prompt_jacobian, test_fit_jlens, test_jlens

In [ ]:
import torch
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)

## From the logit lens to the Jacobian lens

The logit lens reads an intermediate residual stream $h_\ell$ by *skipping* all remaining layers:
$\text{lens}(h_\ell) = W_U\,\mathrm{LN}(h_\ell)$.

The **Jacobian lens** (Gurnee et al., 2026) replaces the skipped layers by a single linear map $J_\ell$ that describes how they *react* to $h_\ell$:

$$J_\ell = \mathbb{E}_{\text{prompt},\; t,\; t' \ge t}\left[\frac{\partial h_{\text{final},t'}}{\partial h_{\ell,t}}\right] \qquad\qquad \text{lens}(h_\ell) = W_U\,\mathrm{LN}(J_\ell\, h_\ell)$$

- $\partial h_{\text{final},t'} / \partial h_{\ell,t}$ is a `(d_model, d_model)` matrix: how a small change of the residual stream at layer $\ell$, position $t$ changes the final residual stream at position $t'$.
- We look at the current position **and all later ones** ($t' \ge t$), because a thought can be said now or later.
- We **average** over many positions and prompts, so $J_\ell$ captures what the model does with $h_\ell$ *in general*, not in one particular context.

Like any lens, it has two halves:
1. **Training**: estimate $J_\ell$ by averaging Jacobians over a text corpus.
2. **Reading**: apply $J_\ell$ to a residual stream and unembed.

You will implement both, and test them on small inputs. (Training a real lens means running this over thousands of positions, which takes about an hour on a GPU.)

In [ ]:
model = HookedTransformer.from_pretrained("gpt2-small")

## Part 1: Training the lens

### Exercise 1: Run the rest of the model

To take derivatives of the final residual stream with respect to $h_\ell$, we first need a function $h_\ell \mapsto h_{\text{final}}$.
TransformerLens can run only part of the model:

```python
model(x, start_at_layer=a, stop_at_layer=b)
```
feeds the residual stream `x` (with a batch dimension) into block `a`, runs blocks `a, ..., b-1`, and returns the residual stream before block `b`.
With `b = model.cfg.n_layers` that is the final residual stream (before the final LayerNorm).

Write `run_from_layer`: it takes the residual stream **after** block `layer`, shape `(seq_len, d_model)`, and returns the final residual stream, shape `(seq_len, d_model)`.

In [ ]:
def run_from_layer(model, resid, layer):
    """(seq_len, d_model) residual after block `layer` -> (seq_len, d_model) final residual stream"""
    ...

In [ ]:
test_run_from_layer(run_from_layer, model)

<details>
<summary><b>Solution</b></summary>

```python
def run_from_layer(model, resid, layer):
    return model(resid[None], start_at_layer=layer + 1, stop_at_layer=model.cfg.n_layers)[0]
```
</details>

### Exercise 2: The Jacobian of one prompt

For one prompt we need the **sum** over all position pairs:

$$S_\ell = \sum_{t}\sum_{t' \ge t} \frac{\partial h_{\text{final},t'}}{\partial h_{\ell,t}} \qquad \text{shape } (d_\text{model}, d_\text{model})$$

Two tricks make this short:
- **Sum the outputs first.** Because attention is causal, $h_{\text{final},t'}$ does not depend on $h_{\ell,t}$ for $t > t'$. So the Jacobian of $\sum_{t'} h_{\text{final},t'}$ with respect to $h_{\ell,t}$ is exactly $\sum_{t' \ge t}$. The condition $t' \ge t$ comes for free.
- **Differentiate with respect to all positions at once.** `torch.autograd.functional.jacobian(f, x, vectorize=True)` returns the Jacobian of `f` at `x`. With `f` returning `(d_model,)` and `x` of shape `(seq_len, d_model)`, you get shape `(d_model, seq_len, d_model)`. Then sum over the input positions.

Write `prompt_jacobian(model, tokens, layer)` returning $S_\ell$.

In [ ]:
def prompt_jacobian(model, tokens, layer):
    """(1, seq_len) tokens -> (d_model, d_model) sum over all pairs t' >= t of d h_final,t' / d h_layer,t"""
    ...

In [ ]:
test_prompt_jacobian(prompt_jacobian, model)

<details>
<summary><b>Solution</b></summary>

```python
def prompt_jacobian(model, tokens, layer):
    _, cache = model.run_with_cache(tokens, remove_batch_dim=True)
    resid = cache[f"blocks.{layer}.hook_resid_post"]
    f = lambda h: run_from_layer(model, h, layer).sum(0)
    J = torch.autograd.functional.jacobian(f, resid, vectorize=True)
    return J.sum(1)
```
</details>

<details>
<summary><b>How does the test know your answer is right?</b></summary>

Computing the full Jacobian is expensive, but multiplying it with a single vector $v$ is cheap: forward-mode differentiation (`torch.func.jvp`) gives $S_\ell v$ in one pass, without ever building $S_\ell$.
The test compares your $S_\ell v$ with that for a random $v$.
It also checks the last layer, where $h_\text{final} = h_\ell$: every pair with $t' = t$ contributes the identity and every other pair contributes zero, so $S_\ell$ must be `seq_len` times the identity.
</details>

### Exercise 3: Average over a corpus

$J_\ell$ is the **average** over all position pairs $(t, t' \ge t)$ of all prompts.
A prompt of length $n$ has $n(n+1)/2$ such pairs.

Write `fit_jlens(model, prompts, layer)`: sum `prompt_jacobian` over the prompts and divide by the total number of pairs.

In [ ]:
def fit_jlens(model, prompts, layer):
    """list of (1, seq_len) token tensors -> (d_model, d_model) J for this layer"""
    ...

In [ ]:
test_fit_jlens(fit_jlens, model)

<details>
<summary><b>Solution</b></summary>

```python
def fit_jlens(model, prompts, layer):
    total = sum(prompt_jacobian(model, tokens, layer) for tokens in prompts)
    n_pairs = sum(tokens.shape[1] * (tokens.shape[1] + 1) / 2 for tokens in prompts)
    return total / n_pairs
```
</details>

## Part 2: Reading with the lens

From the logit lens notebook, here is `unembed` (final LayerNorm, then $W_U$):

In [ ]:
def unembed(model, resid):
    return model.unembed(model.ln_final(resid))

### Exercise 4: The J-lens readout

Write `jlens(model, J, resid, layer)`: apply $J_\ell$ to the residual stream at every position, then unembed.
`J` has shape `(n_layers, d_model, d_model)`: one matrix per layer, as you would get by running `fit_jlens` for every layer.
`resid` has shape `(seq_len, d_model)` with positions as rows, so "$J_\ell h$ for every row" is `resid @ J[layer].T`.

In [ ]:
def jlens(model, J, resid, layer):
    """(seq_len, d_model) residual after block `layer` -> (seq_len, d_vocab) J-lens logits"""
    ...

In [ ]:
test_jlens(jlens, model)

<details>
<summary><b>Solution</b></summary>

```python
def jlens(model, J, resid, layer):
    return unembed(model, resid @ J[layer].T)
```
</details>